# FastAPI from Frictionless Data Packages Devlopment Log

## 2026-02-26 - @jinskeep-morpc

### 1. Create the skeleton for the package using morpc-py and development already started in morpc-purpleair-model as the basis. 

- Had some issues with getting myst pages to work with the github action. 
- The issues was that the ipynbs for demos and devlog were empty, which created an issue when making the pages.
- Added titles to both and it worked.
- See [https://jinskeep-morpc.github.io/fastapi-from-frictionless/](https://jinskeep-morpc.github.io/fastapi-from-frictionless/)


### 2. Creating SQLmodel from frictionless

- I am going to use the schemas from the [morpc-purpleair-model](https://github.com/morpc/morpc-purpleair-model) as the first use case as this is already mostly developed.

> [!NOTE]
> I will be working primarily in the doc folder then moving things over to the package folder when I refactor.

1. I want to create some mock resource files and then combine then into a data package.
    - Create a yaml file for resources. 
    - Work around for creating a resource yaml without actually creating the file. 
        - Try leaving path blank for now and not validate on the creation of the resource?
        - Nevermind, I just need to create the schemas and create the resource file when I have models and database created. 
2. Creating schemas


In [ ]:
import frictionless

deployment_schema = frictionless.Schema('data/deployment.schema.yaml')

In [ ]:
deployment_schema

#### Creating type map for frictionless to pydantic data types

> [!NOTE]
> [Frictionless types](https://datapackage.org/standard/table-schema/#field-types)

> [!NOTE]
> [Pydantic types](https://docs.pydantic.dev/1.10/usage/types/#standard-library-types)

In [ ]:
from frictionless import Schema, extract, fields

extract([["name"], [9]], schema=Schema(fields=[fields.StringField(name='name', format="default")]))

In [ ]:
extract([["name"], ["http://morpc.org"]], schema=Schema(fields=[fields.StringField(name='name', format="uri")]))

In [ ]:
extract([["name"], ["not_a_uri"]], schema=Schema(fields=[fields.StringField(name='name', format="uri")]))

In [ ]:
extract([["name"], ["dataandmaps@morpc.org"]], schema=Schema(fields=[fields.StringField(name='name', format="email")]))

In [ ]:
from uuid import uuid4

extract([["name"], [f"{uuid4()}"]], schema=Schema(fields=[fields.StringField(name='name', format="uuid")]))

In [ ]:
import base64

bb = base64.b64encode(bytes("This is a test of binary strings", 'utf-8'))
extract([["name"], [f"{bb}"]], schema=Schema(fields=[fields.StringField(name='name', format="binary")]))

In [ ]:
extract([["name"], [10.2]], schema=Schema(fields=[fields.NumberField(name='name')]))

In [ ]:
frictionless.settings.DEFAULT_FIELD_CANDIDATES

In [3]:
type_map = {
    "string": {
        "default": "str",
        "email": "EmailStr",
        "uri": "AnyUrl",
        "binary": "bytes",
        "uuid": "UUID"
    },
    "number": {
        "default": "float"
    },
    "integer": {
        "default": "int"
    },
    "boolean": {
        "default": "bool"
    },
    "object": {
        "default": "Json[Any]"
    },
    "array": {
        "default": "List[Any]"
    },
    "datetime": {
        "default": "datetime"
    },
    "date": {
        "default": "date"
    },
    "time": {
        "default": "time"
    },
    "year": {
        "default": "int"
    },
    "duration": {
        "default": "timedelta"
    },
    "geopoint": {
        "default": "Geometry('POINT')"
    },
    "geojson": {
        "default": "Geometry('GEOMETRY')"
    }
}

In [4]:
header = """
from typing import Optional, List
from uuid import UUID
from sqlalchemy import DateTime
from sqlmodel import Field, Relationship, SQLModel
from datetime import date, datetime, timezone, time, timedelta
from pydantic import EmailStr, AnyUrl, Json
from geoalchemy2.types import Geometry

def utcnow():
    '''Returns the current time in UTC.'''
    return datetime.now(timezone.utc)

class TimestampMixin: # https://www.davidmuraya.com/blog/reusable-sqlmodel-mixins/
    '''A mixin to add created_at and updated_at timestamp fields to a model.'''

    created_at: datetime = Field(
        default_factory=utcnow,
        nullable=False,
        sa_type=DateTime(timezone=True)
    )
    updated_at: datetime = Field(
        default_factory=utcnow,
        nullable=False,
        sa_column_kwargs={"onupdate": utcnow},
        sa_type=DateTime(timezone=True)
    )
"""

In [26]:
import os
import frictionless
folder = './data'
schema_paths = [x for x in os.listdir(folder) if x.endswith('schema.yaml')]


In [45]:
models = []
for filename in schema_paths:
    filepath = os.path.join(folder, filename)
    name = filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
    schema = frictionless.Schema(filepath)
    foreign_keys = [x['fields'][0] for x in schema.foreign_keys]

    basemodel_fields = []
    auto_id = "id" in schema.primary_key
    for field in schema.field_names:
        field = schema.get_field(field)

        if (field.name == 'id') & (auto_id == True):
            continue
        else:
            field_string = ""
            field_string += f"{field.name}: "
            field_string += f"{type_map[field.type][field.format]}"  

            if not 'required' in field.constraints:
                field_string += " | None"
                required = False

            if field.name in schema.primary_key:
                field_string += " = Field(primary_key = True)"
            
            if field.name in foriegn_keys:
                field_string += f" = Field({"default=None, " if required else ""}foreign_key='{field.name.replace('_', '.')}')"

            basemodel_fields.append(field_string)

    relationships = []
    for other_filename in schema_paths:
        if other_filename != filename:
            other_filepath = os.path.join(folder, other_filename)
            other_name = other_filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
            other_schema = frictionless.Schema(other_filepath)
            if len(other_schema.foreign_keys) > 0:
                for fk in other_schema.foreign_keys:
                    if fk['reference']['resource'] == name.lower():
                        relationships.append(other_name)

    basemodel_string = f"""class {name}Base(SQLModel):
    {"\n    ".join(basemodel_fields)}
"""

    tablemodel_string = f"""class {name}({name}Base, TimestampMixin, table=True):
"""
    if auto_id == True:
        tablemodel_string += "    id: int | None = Field(default=None, primary_key=True)\n"
    if len(relationships) > 0:
        for relationship in relationships:
            tablemodel_string += f"    {relationship.lower()}s: list['{relationship}'] | None = Relationship(back_populates='{name.lower()}')\n"
    if (auto_id == False) & (len(relationships) == 0):
        tablemodel_string += '    pass\n'


    createmodel_string = f"""class {name}Create({name}Base):
    pass
"""
    
    updatemodel_string = f"""class {name}Update({name}Base):\n"""
    for field in basemodel_fields:
        if not 'primary_key' in field:
            if " = " in field:
                field = field.split(" = ")[0]
            if not ' | None' in field:
                updatemodel_string += f"    {field} | None\n"
            else:
                updatemodel_string +=  f"    {field}\n"

    publicmodel_string = f"""class {name}Public({name}Base):{"\n    id: int" if auto_id == True else ''}
    created_at: datetime
    updated_at: datetime
"""
    
    relationshipsmodel_string = f""

    if len(foreign_keys) > 0:
        relationshipsmodel_string += f"""class {name}PublicWithAll({name}Public):\n"""
        for fk in foreign_keys:
            relationshipsmodel_string += f"    {fk.split("_")[0]}s: list['{fk.split('_')[0].capitalize()}'] | None\n"


    model_file = f"""
## {name} models
{basemodel_string}
{tablemodel_string}
{createmodel_string}
{publicmodel_string}
{relationshipsmodel_string}
{updatemodel_string}
    """
    models.append(model_file)



In [47]:
with open('models.py', 'w') as file:
    file.write("".join([header] + models))

In [19]:
filename = 'test.db'
database_string = f"""
from sqlmodel import SQLModel, create_engine

sqlite_filename = '{filename}'
sqlite_url = f"sqlite:///{{sqlite_filename}}"

connect_args = {{'check_same_thread': False}}
engine = create_engine(sqlite_url, echo=True, connect_args=connect_args)

def create_db_and_tables():
    SQLModel.metadata.create_all(engine)
"""
print(database_string)
with open('database.py', 'w') as file:
    file.write(database_string)


from sqlmodel import SQLModel, create_engine

sqlite_filename = 'test.db'
sqlite_url = f"sqlite:///{sqlite_filename}"

connect_args = {'check_same_thread': False}
engine = create_engine(sqlite_url, echo=True, connect_args=connect_args)

def create_db_and_tables():
    SQLModel.metadata.create_all(engine)



In [20]:
app_header = """
# app.py
from fastapi import Depends, FastAPI, HTTPException, Query
import fastapi
from fastapi_querybuilder import QueryBuilder
from sqlalchemy import text
from sqlmodel import Session, select
from .database import create_db_and_tables, engine
from .models import *
from sqlalchemy.ext.asyncio import AsyncSession

# Initiate app
app = FastAPI()

# Dependencies
@app.on_event('startup')
def on_startup():
    create_db_and_tables()

def get_session():
    with Session(engine) as session:
        yield session"""

In [48]:
import os
folder = './data'
schema_paths = [x for x in os.listdir(folder) if x.endswith('schema.yaml')]
endpoints = []

for filename in schema_paths:
    filepath = os.path.join(folder, filename)
    name = filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
    schema = frictionless.Schema(filepath)
    foreign_keys = [x['fields'][0] for x in schema.foreign_keys]
    
    post_string = f"""
# {name} requests
@app.post('/{name.lower()}s/', response_model={name}Public)
def create_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}: {name}Create):
    {name.lower()} = {name}.model_validate({name.lower()})
    session.add({name.lower()})
    session.commit()
    session.refresh({name.lower()})
    return {name.lower()}"""
    
    relationships = []
    for other_filename in schema_paths:
        if other_filename != filename:
            other_filepath = os.path.join(folder, other_filename)
            other_name = other_filename.split('.')[0].replace('-', ' ').title().replace(' ', '')
            other_schema = frictionless.Schema(other_filepath)
            if len(other_schema.foreign_keys) > 0:
                for fk in other_schema.foreign_keys:
                    if fk['reference']['resource'] == name.lower():
                        relationships.append(other_name)

    pk = schema.primary_key[0]

    getall_string = f"""
@app.get('/{name.lower()}s/', response_model=list[{f'{name}PublicWithAll' if len(foreign_keys)>0 else f'{name}Public'}])
def read_{name.lower()}s(*, session: Session = Depends(get_session)):
    {name.lower()}s = session.exec(select({name})).all()
    return {name.lower()}s"""


    if len(foreign_keys) > 0:
        query_string = f"""
@app.get('/{name.lower()}s/query', response_model=list[{name}PublicWithAll])
async def query_{name.lower()}s(*, session: AsyncSession = Depends(get_session), query=QueryBuilder({name})):
    {name.lower()}s = session.execute(query)
    return {name.lower()}s.scalars().all()"""
    else:
        query_string = ""
    
    get_string = f"""
@app.get('/{name.lower()}s/{{{name.lower()}_{pk}}}', response_model={f'{name}PublicWithAll' if len(foreign_keys)>0 else f'{name}Public'})
def read_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}_{pk}: str):
    {name.lower()} = session.get({name}, {name.lower()}_{pk})
    if not {name.lower()}:
        raise HTTPException(status_code=404, detail='{name} not found.')
    return {name.lower()}"""
    
    update_string = f"""
@app.patch('/{name.lower()}s/{{{name.lower()}_{pk}}}', response_model={name}Public)
def update_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}_{pk}: str, {name.lower()}: {name}Update):
    db_{name.lower()} = session.get({name}, {name.lower()}_{pk})
    if not db_{name.lower()}:
        raise HTTPException(status_code=404, detail=f'{name} {{{name.lower()}_{pk}}} not found.')
    {name.lower()}_data = {name.lower()}.model_dump(exclude_unset=True)
    db_{name.lower()}.sqlmodel_update({name.lower()}_data)
    session.add(db_{name.lower()})
    session.commit()
    session.refresh(db_{name.lower()})
    return db_{name.lower()}"""
    
    delete_string = f"""
@app.delete('/{name.lower()}s/{{{name.lower()}_{pk}}}')
def delete_{name.lower()}(*, session: Session = Depends(get_session), {name.lower()}_{pk}: str):
    {name.lower()} = session.get({name}, {name.lower()}_{pk})
    if not {name.lower()}:
        raise HTTPException(status_code=404, detail=f'{name} {{{name.lower()}_{pk}}} not found.')
    session.delete({name.lower()})
    session.commit()
    return {{'ok': True}}"""

    endpoint_file = f"""
    {post_string}
    {getall_string}
    {get_string}
    {query_string}
    {update_string}
    {delete_string}
    """

    endpoints.append(endpoint_file)



In [49]:
endpoints

["\n    \n# Sensor requests\n@app.post('/sensors/', response_model=SensorPublic)\ndef create_sensor(*, session: Session = Depends(get_session), sensor: SensorCreate):\n    sensor = Sensor.model_validate(sensor)\n    session.add(sensor)\n    session.commit()\n    session.refresh(sensor)\n    return sensor\n    \n@app.get('/sensors/', response_model=list[SensorPublic])\ndef read_sensors(*, session: Session = Depends(get_session)):\n    sensors = session.exec(select(Sensor)).all()\n    return sensors\n    \n@app.get('/sensors/{sensor_macaddr}', response_model=SensorPublic)\ndef read_sensor(*, session: Session = Depends(get_session), sensor_macaddr: str):\n    sensor = session.get(Sensor, sensor_macaddr)\n    if not sensor:\n        raise HTTPException(status_code=404, detail='Sensor not found.')\n    return sensor\n    \n    \n@app.patch('/sensors/{sensor_macaddr}', response_model=SensorPublic)\ndef update_sensor(*, session: Session = Depends(get_session), sensor_macaddr: str, sensor: Sen

In [50]:
with open('app.py', 'w') as file:
    file.write("".join([app_header] + endpoints))